In [ ]:
import os
import sys
from pathlib import Path

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"

os.environ["HADOOP_HOME"] = "/opt/hadoop-3.3.6"
os.environ["HADOOP_CONF_DIR"] = "/opt/hadoop-3.3.6/etc/hadoop"
os.environ["YARN_CONF_DIR"] = os.environ["HADOOP_CONF_DIR"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"

derby_path = (BASE / "metastore" / "metastore_db").as_posix()

print("Metastore:", derby_path)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("afp_v2_validacion_hive")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "10.0.2.15")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config(
        "javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.sql.warehouse.dir",
        "hdfs://localhost:9000/user/junior/datalake_afp_v2/catalog/warehouse"
    )
    .enableHiveSupport()
    .getOrCreate()
)

print("MASTER:", spark.sparkContext.master)
print("CATALOGO:", spark.conf.get("spark.sql.catalogImplementation"))

In [ ]:
tablas = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "tabla_analitica_mensual"
]

print("BASES DE DATOS:")
spark.sql("SHOW DATABASES").show()

print("TABLAS PERSISTENTES:")
spark.sql("""
SHOW TABLES IN afp_analytics
""").show(truncate=False)

print("CONTEO DE REGISTROS:")
for nombre in tablas:
    total = spark.sql(
        f"SELECT COUNT(*) AS total "
        f"FROM afp_analytics.{nombre}"
    ).first()["total"]

    print(nombre, ":", total)

In [ ]:
spark.sql("""
SHOW PARTITIONS afp_analytics.bcrp_series_largas
""").show(10, truncate=False)